In [ ]:
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers.modeling_outputs import CausalLMOutput

from transformers import Trainer, TrainingArguments

from PIL import Image
from dataclasses import dataclass

from torchvision import transforms

import json
import os
from typing import Dict, List, Optional, Tuple

import regex
from transformers import PreTrainedTokenizer

# --- ADDED FOR STAGE-2 ---
import numpy as np                       # array ops for augmentation in ImagePreprocessor
from transformers import TrainerCallback  # used by the CER eval callback


# Image encoder

### Preprocessor

In [ ]:
# Encoder model utils

def random_masking(x, mask_ratio):
    B, N, D = x.shape

    len_keep = int(N * (1 - mask_ratio))

    noise = torch.rand(B, N, device=x.device)

    ids_shuffle = torch.argsort(noise, dim=1)
    ids_restore = torch.argsort(ids_shuffle, dim=1)

    ids_keep = ids_shuffle[:, :len_keep]

    x_masked = torch.gather(
        x,
        dim=1,
        index=ids_keep.unsqueeze(-1).repeat(1, 1, D)
    )

    mask = torch.ones(B, N, device=x.device)
    mask[:, :len_keep] = 0

    mask = torch.gather(mask, dim=1, index=ids_restore)

    return x_masked, mask, ids_restore, ids_keep




def patchify(images, patch_size):
    """
    images: B, C, H, W
    returns: B, N, patch_dim
    """
    B, C, H, W = images.shape

    assert H % patch_size == 0
    assert W % patch_size == 0

    h = H // patch_size
    w = W // patch_size

    patches = images.reshape(
        B, C, h,
        patch_size,
        w,
        patch_size
    )

    patches = torch.einsum('nchpwq->nhwpqc', patches)

    patches = patches.reshape(
        B, h * w,
        patch_size * patch_size * C
    )

    return patches


def get_2d_sinusoidal_encoding(h_patches, w_patches, embed_dim):
    assert embed_dim % 4 == 0  # split evenly between h and w dims

    def sinusoid(length, dim):
        positions = torch.arange(length).unsqueeze(1)          # L, 1
        dims = torch.arange(0, dim, 2).unsqueeze(0)            # 1, D/2
        freqs = 1.0 / (10000 ** (dims / dim))
        args = positions * freqs                                # L, D/2
        emb = torch.cat([args.sin(), args.cos()], dim=-1)      # L, D
        return emb

    h_enc = sinusoid(h_patches, embed_dim // 2)  # H, D/2
    w_enc = sinusoid(w_patches, embed_dim // 2)  # W, D/2

    # Broadcast and combine
    h_enc = h_enc.unsqueeze(1).repeat(1, w_patches, 1)  # H, W, D/2
    w_enc = w_enc.unsqueeze(0).repeat(h_patches, 1, 1)  # H, W, D/2

    encoding = torch.cat([h_enc, w_enc], dim=-1)         # H, W, D
    return encoding.view(h_patches, w_patches, embed_dim)  # N, D




class ImagePreprocessor:
    """
    Preprocesses the image before encoding the image
    1) Change the image to gray scale
    2) Resize the image
    3) Pad the image to the nearest multiple of patch size
    4) Normalize the image
    """
    def __init__(self, image_height: int, max_image_width: int, patch_size: int, augment_fn=None):
        # CHANGED FOR STAGE-2: added `augment_fn`. When set (train split only),
        # augmentation runs on the RESIZED 64px-tall crop (see _transform), not the
        # full-res source -> most of the CPU cost of the expensive augraphy effects
        # (DirtyRollers/DirtyDrum/BleedThrough) disappears, and we augment exactly
        # the pixels the model sees. eval uses augment_fn=None.
        assert image_height % patch_size == 0, "Image height should be a multiple of patch size"
        self.image_height = image_height
        self.max_image_width = max_image_width
        self.patch_size = patch_size
        self.augment_fn = augment_fn
        self.to_tensor = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5])
        ])

    def _transform(self, img: Image.Image) -> torch.Tensor:
        # CHANGED FOR STAGE-2: grayscale conversion moved to AFTER augmentation, so
        # augraphy receives a 3-channel image (its tested path) at the small scale.

        # Calculate the resize target for the image while preserving the aspect ratio
        img_w, img_h = img.size
        scale_factor = self.image_height/img_h
        if scale_factor * img_w > self.max_image_width:
            scale_factor = self.max_image_width/img_w
            target_size = (int(scale_factor * img_h), self.max_image_width)
            diff = self.image_height - target_size[0]
            pad_t, pad_b = diff//2, diff - diff//2
            pad_l, pad_r = 0, 0
        else:
            target_size = (self.image_height, int(scale_factor*img_w))
            # Find the nearest multiple of patch size for padding
            diff = (-target_size[1]) % self.patch_size
            pad_l, pad_r = 0, diff
            pad_t, pad_b = 0, 0

        img = transforms.Resize(target_size)(img)
        img = transforms.Pad((pad_l, pad_t, pad_r, pad_b), fill=255)(img)

        # CHANGED FOR STAGE-2: augment the small (resized) crop, train split only.
        if self.augment_fn is not None:
            arr = np.array(img.convert('RGB'))            # (H, W, 3) uint8 for augraphy
            arr = self.augment_fn(arr)
            img = Image.fromarray(np.asarray(arr, dtype=np.uint8))

        img = img.convert('L')                            # grayscale for the model
        return self.to_tensor(img)

    def __call__(self, img: Image.Image) -> torch.Tensor:
        return self._transform(img)

### Model

In [ ]:
@dataclass
class ViTConfig:
    embed_dim: int = 512
    num_heads: int = 8
    dropout: float = 0.2
    hidden_layer_size: int = 1024
    num_blocks: int = 8
    patch_size: int = 16
    image_height: int = 64
    max_image_width:int = 1024




class TransformerBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, hidden_layer_size: int, dropout: float):
        super().__init__()
        self.multi_head_attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_layer_size),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_layer_size, embed_dim),
            nn.Dropout(dropout)
        )
        self.layer_norm1 = nn.LayerNorm(embed_dim)
        self.layer_norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x, key_padding_mask = None):
        # query, key, value - B, T, D
        norm_x = self.layer_norm1(x)
        ctx_embed, _ = self.multi_head_attention(
            norm_x, norm_x, norm_x,
            key_padding_mask=key_padding_mask,
            need_weights=False)
        ctx_embed = ctx_embed + x

        norm_ctx_embed = self.layer_norm2(ctx_embed)
        norm_ctx_embed = self.mlp(norm_ctx_embed)
        norm_ctx_embed = norm_ctx_embed + ctx_embed
        return norm_ctx_embed



class ViTEncoder(nn.Module):
    def __init__(self, vit_config: ViTConfig):
        super().__init__()
        self.image_embedding = nn.Conv2d(in_channels=1, out_channels=vit_config.embed_dim, kernel_size=vit_config.patch_size, stride=vit_config.patch_size)

        enc = get_2d_sinusoidal_encoding(
            vit_config.image_height // vit_config.patch_size,
            vit_config.max_image_width // vit_config.patch_size,
            vit_config.embed_dim
        )
        self.register_buffer("positional_encoding", enc)

        self.transformer_blocks = nn.ModuleList(
            [TransformerBlock(vit_config.embed_dim, vit_config.num_heads, vit_config.hidden_layer_size, vit_config.dropout)
              for _ in range(vit_config.num_blocks)]
        )
        self.layer_norm = nn.LayerNorm(vit_config.embed_dim)

    def forward(self, x: torch.Tensor, padding_mask: torch.Tensor, mask_ratio: float | None = None):
        # x -> B, C, H, W
        # print("input: ", x.shape)
        embeds = self.image_embedding(x)
        # print("embeds: ", embeds.shape)
        B, D, hp, wp = embeds.shape
        embeds = embeds.flatten(2).transpose(1, 2)
        pos_encodings = self.positional_encoding[:hp, :wp, :].reshape(hp*wp, D)
        embeds = embeds + pos_encodings

        if mask_ratio is not None:
            embeds, mask, restore_ids, ids_keep = random_masking(embeds, mask_ratio)
        else:
            mask, restore_ids, ids_keep = None, None, None

        if ids_keep is not None:
            visible_padding_mask = torch.gather(
                padding_mask, dim=1,
                index=ids_keep
            )
        else:
            visible_padding_mask = padding_mask

        visible_padding_mask = visible_padding_mask.bool()

        ctx_embeds = embeds
        for block in self.transformer_blocks:
            ctx_embeds = block(ctx_embeds, visible_padding_mask)

        ctx_embeds = self.layer_norm(ctx_embeds)
        return ctx_embeds, mask, restore_ids




class ViTDecoder(nn.Module):
    def __init__(self, config: ViTConfig, encoder_dim: int):
        super().__init__()
        self.embedding_layer = nn.Linear(encoder_dim, config.embed_dim)

        self.h_full = config.image_height // config.patch_size
        enc = get_2d_sinusoidal_encoding(
            config.image_height // config.patch_size,
            config.max_image_width // config.patch_size,
            config.embed_dim
        )
        self.register_buffer("positional_encoding", enc)

        self.mask_token = nn.Parameter(torch.zeros(1, 1, config.embed_dim))

        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(config.embed_dim, config.num_heads, config.hidden_layer_size, config.dropout)
            for _ in range(config.num_blocks)
        ])
        self.decoder_norm = nn.LayerNorm(config.embed_dim)

        self.output_projection = nn.Linear(config.embed_dim, config.patch_size ** 2)


    def forward(self, latent: torch.Tensor, restore_ids: torch.Tensor, padding_mask: torch.Tensor):
        x = self.embedding_layer(latent)

        B, L, D = x.shape
        N = restore_ids.shape[1]

        mask_tokens = self.mask_token.repeat(B, N-L, 1)

        _x = torch.cat([x, mask_tokens], dim=1)
        _x = torch.gather(
            _x, dim=1,
            index=restore_ids.unsqueeze(-1).repeat(1, 1, D)
        )
        wp = N // self.h_full
        _x = _x + self.positional_encoding[:, :wp ,:].reshape(N, D)
        for block in self.transformer_blocks:
            _x = block(_x, padding_mask.bool())
        _x = self.decoder_norm(_x)
        proj = self.output_projection(_x)
        return proj

class MaskedAutoEncoder(nn.Module):
    def __init__(self, encoder_config: ViTConfig, decoder_config: ViTConfig, mask_ratio: float, norm_pix_loss: bool = True):
        super().__init__()

        self.mask_ratio = mask_ratio
        self.norm_pix_loss = norm_pix_loss
        self.patch_size = encoder_config.patch_size
        self.encoder_model = ViTEncoder(encoder_config)
        self.decoder_model = ViTDecoder(decoder_config, encoder_config.embed_dim)

    def forward(self, images: torch.Tensor, padding_masks: torch.Tensor):
        latent, mask, restore_ids = self.encoder_model(images, padding_masks,  self.mask_ratio)

        pred = self.decoder_model(latent, restore_ids, padding_masks)

        target = patchify(images, self.patch_size)          # (B, N, patch_size**2)

        if self.norm_pix_loss:
            mean = target.mean(dim=-1, keepdim=True)
            var = target.var(dim=-1, keepdim=True)
            target = (target - mean) / (var + 1e-6).sqrt()

        loss = ((target - pred) ** 2).mean(dim=-1)

        valid_patch_mask = (~padding_masks.bool()).float()
        effective_mask = mask * valid_patch_mask
        loss = (loss * effective_mask).sum() / effective_mask.sum()

        return {"loss": loss, "logits": pred, "mask": mask}

# Text Decoder

### Tokenizer


In [ ]:

class TeluguGraphemeTokenizer(PreTrainedTokenizer):
    """HuggingFace-compatible grapheme-cluster tokenizer for Telugu.

    Parameters
    ----------
    vocab : dict[str, int]
        Mapping from grapheme string → token ID.  Must include all
        ``SPECIAL_TOKENS_LIST`` entries.
    add_bos_token : bool
        Automatically prepend ``[BOS]`` on ``encode()``.
    add_eos_token : bool
        Automatically append ``[EOS]`` on ``encode()``.
    """

    vocab_files_names = {"vocab_file": "vocab.json"}  # ← fix 1
    model_input_names = ["input_ids", "attention_mask"]  # ← fix 2

    def __init__(
            self,
            vocab_file: Optional[str] = None,
            vocab_list: Optional[list[str]] = None,
            add_bos_token: bool = True,
            add_eos_token: bool = True,
            pad_token: str = "[PAD]",
            unk_token: str = "[UNK]",
            bos_token: str = "[BOS]",
            eos_token: str = "[EOS]",
            mask_token: str = "[MASK]",
            **kwargs,
    ):
        vocab = {}

        if vocab_file is not None:
            with open(vocab_file, encoding="utf-8") as f:
                vocab = json.load(f)

        if not vocab and vocab_list is not None:
            for grapheme in vocab_list:
                vocab[grapheme] = len(vocab)

        if not vocab:
            raise AssertionError("Either `vocab_file` or `vocab_list` has to be given.")

        self.SPECIAL_TOKENS_LIST = [pad_token, unk_token, bos_token, eos_token, mask_token]

        self.grapheme_pattern = regex.compile(r'\X')

        for tok in self.SPECIAL_TOKENS_LIST:
            if tok not in vocab:
                vocab[tok] = len(vocab)

        self.vocab = vocab  # ← must be BEFORE super().__init__()
        self._inv_vocab = {v: k for k, v in vocab.items()}
        self.add_bos_token = add_bos_token
        self.add_eos_token = add_eos_token
        self.UNK = unk_token

        super().__init__(
            pad_token=pad_token,
            unk_token=unk_token,
            bos_token=bos_token,
            eos_token=eos_token,
            mask_token=mask_token,
            add_bos_token=add_bos_token,
            add_eos_token=add_eos_token,
            padding_side="right",
            model_max_length=4096,
            **kwargs,
        )

    @property
    def vocab_size(self) -> int:
        return len(self.vocab)

    def get_vocab(self) -> Dict[str, int]:
        return dict(self.vocab)

    def _tokenize(self, text: str, **kwargs) -> List[str]:
        tokens = []
        for grapheme in self.grapheme_pattern.findall(text):
            if grapheme in self.vocab:
                tokens.append(grapheme)
            else:
                for codepoint in grapheme:
                    tokens.append(codepoint)
        return tokens

    def _convert_token_to_id(self, token: str) -> int:
        return self.vocab.get(token, self.vocab.get(self.UNK, 1))

    def _convert_id_to_token(self, index: int) -> str:
        return self._inv_vocab.get(index, self.UNK)

    def convert_tokens_to_string(self, tokens: List[str]) -> str:
        cleaned = [t for t in tokens if t not in self.SPECIAL_TOKENS_LIST]
        return "".join(cleaned)

    def build_inputs_with_special_tokens(self, token_ids_0, token_ids_1=None):
        bos = [self.bos_token_id] if self.add_bos_token else []
        eos = [self.eos_token_id] if self.add_eos_token else []
        out = bos + token_ids_0 + eos
        if token_ids_1 is not None:
            out += bos + token_ids_1 + eos
        return out

    def get_special_tokens_mask(self, token_ids_0, token_ids_1=None, already_has_special_tokens=False):
        if already_has_special_tokens:
            return super().get_special_tokens_mask(token_ids_0, token_ids_1, already_has_special_tokens=True)
        bos = [1] if self.add_bos_token else []
        eos = [1] if self.add_eos_token else []
        res = bos + [0] * len(token_ids_0) + eos
        if token_ids_1 is not None:
            res += bos + [0] * len(token_ids_1) + eos
        return res

    def save_vocabulary(
            self,
            save_directory: str,
            filename_prefix: Optional[str] = None,
    ) -> Tuple[str]:
        os.makedirs(save_directory, exist_ok=True)
        fname = (filename_prefix + "-" if filename_prefix else "") + "vocab.json"
        vocab_path = os.path.join(save_directory, fname)
        with open(vocab_path, "w", encoding="utf-8") as f:
            json.dump(self.vocab, f, ensure_ascii=False, indent=2)
        return (vocab_path,)



## Model

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 2048
    embed_dim: int = 512
    hidden_dim: int = 2048
    num_heads: int = 8
    num_layers: int = 12
    ctx_len: int = 1024
    dropout: float = 0.1


def calculate_positional_encodings(positions: torch.Tensor, embed_dim: int):
    i = torch.arange(embed_dim // 2, dtype=torch.float32)
    div_term = 10000 ** (2 * i / embed_dim)          # (D/2,)
    pos = positions.float().unsqueeze(1)              # (T, 1)
    args = pos / div_term                             # (T, D/2)
    enc = torch.zeros(len(positions), embed_dim)
    enc[:, 0::2] = torch.sin(args)
    enc[:, 1::2] = torch.cos(args)
    return enc


class SwiGLU(nn.Module):
    """Implement the SwiGLU activation function"""
    def __init__(self, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.gate_proj = nn.Linear(embed_dim, hidden_dim, bias=False)
        self.up_proj = nn.Linear(embed_dim, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, embed_dim, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class MultiHeadAttention(nn.Module):
    """Implement multi head attention"""
    def __init__(self, embed_dim: int, num_heads: int, dropout: float):
        super().__init__()
        assert embed_dim % num_heads == 0, \
            f"embed_dim ({embed_dim}) must be divisible by num_heads ({num_heads})"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.dropout   = dropout

        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, attn_mask=None):
        # attn_mask: (B, 1, T, T) boolean — True means KEEP, False means MASK OUT
        B, T, _ = query.shape
        _, S, _ = key.shape

        queries = self.q_proj(query).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        keys    = self.k_proj(key).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        values  = self.v_proj(value).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)

        dropout_p = self.dropout if self.training else 0.0
        ctx_embeds = F.scaled_dot_product_attention(
            queries, keys, values,
            dropout_p=dropout_p,
            attn_mask=attn_mask,
            is_causal=False
        )

        ctx_embeds = ctx_embeds.transpose(1, 2).reshape(B, T, self.embed_dim)
        return self.out_proj(ctx_embeds)


class GPTTransformerBlock(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.attention_layer = MultiHeadAttention(config.embed_dim, config.num_heads, config.dropout)
        self.mlp = nn.Sequential(
            SwiGLU(config.embed_dim, config.hidden_dim),
            nn.Dropout(config.dropout)
        )
        self.layer_norm1 = nn.LayerNorm(config.embed_dim)
        self.layer_norm2 = nn.LayerNorm(config.embed_dim)

    def forward(self, x: torch.Tensor, attn_mask = None):
        # x -> B, T, D
        normed = self.layer_norm1(x)
        x = x + self.attention_layer(normed, normed, normed, attn_mask=attn_mask)
        x = x + self.mlp(self.layer_norm2(x))
        return x



class GPTModel(nn.Module):
    _keys_to_ignore_on_save = None
    def __init__(self, config: GPTConfig, pad_index: int):
        super().__init__()
        self.pad_index = pad_index
        self.embedding_layer = nn.Embedding(config.vocab_size, config.embed_dim)
        self.register_buffer("positional_encodings", calculate_positional_encodings(torch.arange(config.ctx_len), config.embed_dim))
        self.transformer_blocks = nn.ModuleList([
            GPTTransformerBlock(config) for _ in range(config.num_layers)
        ])
        self.layer_norm = nn.LayerNorm(config.embed_dim)
        self.lm_head = nn.Linear( config.embed_dim, config.vocab_size, bias=False)

    def _build_attn_mask(self, input_ids, attention_mask):
        """
        Builds a combined boolean causal + padding mask.
        SDPA expects: True = attend, False = ignore.
        Shape: (B, 1, T, T)
        """
        B, T = input_ids.shape
        device = input_ids.device
        # Causal mask: upper triangle is False (masked), lower triangle True
        causal = torch.ones(T, T, dtype=torch.bool, device=device).tril()  # (T, T)
        if attention_mask is not None:
            # attention_mask: (B, T), 1=real token, 0=pad
            # Expand to (B, 1, 1, T) so it broadcasts over query positions
            pad_mask = attention_mask.bool().unsqueeze(1).unsqueeze(2)      # (B, 1, 1, T)
            combined = causal.unsqueeze(0).unsqueeze(0) & pad_mask          # (B, 1, T, T)
        else:
            combined = causal.unsqueeze(0).unsqueeze(0)                     # (1, 1, T, T)
        return combined

    def forward(self, input_ids, attention_mask=None, labels = None):
        # input_ids -> (B, T)
        embeds = self.embedding_layer(input_ids)
        T = input_ids.shape[1]
        embeds = embeds + self.positional_encodings[:T].unsqueeze(0)
        attn_mask = self._build_attn_mask(input_ids, attention_mask)
        for block in self.transformer_blocks:
            embeds = block(embeds, attn_mask=attn_mask)
        embeds = self.layer_norm(embeds)
        logits = self.lm_head(embeds)
        loss = None

        # Always calculate the loss
        # shift for causal LM
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=self.pad_index
        )
        return CausalLMOutput(
            loss=loss,
            logits=logits
        )

# Encoder Decoder Model

In [ ]:
class DecoderTransformerBlock(nn.Module):
    """Transformer block with cross attention"""
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.attention_layer = MultiHeadAttention(config.embed_dim, config.num_heads, dropout=config.dropout)
        self.cross_attention_layer = MultiHeadAttention(config.embed_dim, config.num_heads, dropout=config.dropout)

        self.mlp = nn.Sequential(
            SwiGLU(config.embed_dim, config.hidden_dim),
            nn.Dropout(config.dropout)
        )

        self.layer_norm1 = nn.LayerNorm(config.embed_dim)
        self.layer_norm1_5 = nn.LayerNorm(config.embed_dim)
        self.layer_norm2 = nn.LayerNorm(config.embed_dim)

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor, attn_mask = None, padding_mask = None):
        normed = self.layer_norm1(x)
        attn_out = self.attention_layer(normed, normed, normed, attn_mask)
        x = x + attn_out

        cross_in = self.layer_norm1_5(x)
        cross_out = self.cross_attention_layer(cross_in, encoder_output, encoder_output, padding_mask)
        x = x + cross_out

        mlp_in = self.layer_norm2(x)
        mlp_out = self.mlp(mlp_in)
        x = x + mlp_out

        return x





class TextDecoder(nn.Module):
    """A text decoder model of the transformer model"""
    _keys_to_ignore_on_save = None
    def __init__(self, config: GPTConfig, pad_index: int):
        super().__init__()
        self.pad_index = pad_index
        self.embedding_layer = nn.Embedding(config.vocab_size, config.embed_dim)
        self.register_buffer("positional_encodings", calculate_positional_encodings(torch.arange(config.ctx_len), config.embed_dim))
        self.transformer_blocks = nn.ModuleList([
            DecoderTransformerBlock(config) for _ in range(config.num_layers)
        ])
        self.layer_norm = nn.LayerNorm(config.embed_dim)
        self.lm_head = nn.Linear( config.embed_dim, config.vocab_size, bias=False)


    def _build_causal_attn_mask(self, input_ids, padding_mask):
        """
        Builds a combined boolean causal + padding mask.
        SDPA expects: True = attend, False = ignore.
        Shape: (B, 1, T, T)
        """
        B, T = input_ids.shape
        device = input_ids.device
        # Causal mask: upper triangle is False (masked), lower triangle True
        causal = torch.ones(T, T, dtype=torch.bool, device=device).tril()  # (T, T)
        if padding_mask is not None:
            # padding_mask: (B, T), 1=real token, 0=pad
            # Expand to (B, 1, 1, T) so it broadcasts over query positions
            pad_mask = padding_mask.bool().unsqueeze(1).unsqueeze(2)      # (B, 1, 1, T)
            combined = causal.unsqueeze(0).unsqueeze(0) & pad_mask          # (B, 1, T, T)
        else:
            combined = causal.unsqueeze(0).unsqueeze(0)                     # (1, 1, T, T)
        return combined

    def forward(self, input_ids, encoder_output, text_padding_mask=None, img_text_padding_mask = None, labels = None):
        # input_ids -> (B, T)
        embeds = self.embedding_layer(input_ids)
        T = input_ids.shape[1]
        embeds = embeds + self.positional_encodings[:T].unsqueeze(0)
        causal_attn_mask = self._build_causal_attn_mask(input_ids, text_padding_mask)
        for block in self.transformer_blocks:
            embeds = block(embeds, encoder_output, attn_mask = causal_attn_mask, padding_mask = img_text_padding_mask)
        embeds = self.layer_norm(embeds)
        logits = self.lm_head(embeds)
        loss = None

        # Always calculate the loss
        # shift for causal LM
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=self.pad_index
        )
        return CausalLMOutput(
            loss=loss,
            logits=logits
        )


class EncoderDecoder(nn.Module):
    """An Image encoder and text decoder based transformer model"""
    def __init__(self, encoder_config: ViTConfig, decoder_config: GPTConfig, pad_index: int):
        super().__init__()
        self.encoder_model = ViTEncoder(encoder_config)
        self.decoder_model = TextDecoder(decoder_config, pad_index)
        # NEW FOR STAGE-2: stored so generate() can build the image padding mask.
        self.patch_size = encoder_config.patch_size
        self.img_height = encoder_config.image_height

    def forward(self, pixel_values, input_ids, img_padding_mask = None, text_padding_mask=None, return_loss=True):
        # x -> B, C, H, W
        encoder_output, _, _ = self.encoder_model(pixel_values, padding_mask=img_padding_mask, mask_ratio = None)

        if img_padding_mask is not None:
            cross_key_mask = (~img_padding_mask.bool()).unsqueeze(1).unsqueeze(2)
        else:
            cross_key_mask = None
        decoder_output = self.decoder_model(input_ids, encoder_output, text_padding_mask, cross_key_mask, None)
        return decoder_output

    # ------------------------------------------------------------------
    # NEW FOR STAGE-2: greedy autoregressive decode, for CER evaluation.
    # Single-sample (B=1). Correct-by-construction (full recompute per step);
    # slow, but only used on a small eval subset periodically.
    # ------------------------------------------------------------------
    @torch.no_grad()
    def generate(self, pixel_values, bos_id, eos_id, max_new_tokens=256):
        self.eval()
        device = pixel_values.device
        B, C, H, W = pixel_values.shape
        hp, wp = H // self.patch_size, W // self.patch_size
        img_pad = torch.zeros(B, hp * wp, dtype=torch.float, device=device)  # all valid
        enc_out, _, _ = self.encoder_model(pixel_values, padding_mask=img_pad, mask_ratio=None)
        cross_key_mask = (~img_pad.bool()).unsqueeze(1).unsqueeze(2)
        ids = torch.full((B, 1), bos_id, dtype=torch.long, device=device)
        ctx = self.decoder_model.positional_encodings.shape[0]
        for _ in range(min(max_new_tokens, ctx - 1)):
            out = self.decoder_model(ids, enc_out, text_padding_mask=None,
                                     img_text_padding_mask=cross_key_mask, labels=None)
            nxt = out.logits[:, -1, :].argmax(-1, keepdim=True)
            ids = torch.cat([ids, nxt], dim=1)
            if nxt.item() == eos_id:
                break
        return ids[0].tolist()


## Create the model, load the weights and freeze the pre-trained weights

In [ ]:
# ============================================================================
# CHANGED FOR STAGE-2 (full fine-tuning) — ENTIRE CELL REPLACED
# WHAT : was "load pretrained text + pretrained MAE, copy weights, FREEZE all but
#        cross-attention". Now: rebuild the model and load the STAGE-1 checkpoint,
#        with EVERYTHING trainable.
# WHY  : stage-2 continues from stage-1's already-aligned cross-attention and
#        unfreezes all layers, so we neither re-init from the two pretrained models
#        nor freeze anything.
# ============================================================================
tokenizer = TeluguGraphemeTokenizer(vocab_file="/kaggle/input/models/harshadesaraju1999/telugu-grapheme-tokenizer/transformers/default/1/telugu-vocab.json")
print(tokenizer.pad_token_id)

# --- Configs: MUST match stage-1 exactly so the checkpoint loads cleanly ---
decoder_config = GPTConfig(
    vocab_size=len(tokenizer),
    embed_dim=512,
    hidden_dim=1368,
    num_heads=8,
    num_layers=16,
    ctx_len=256,
    dropout=0.1,
)
img_encoder_cfg = ViTConfig(
    embed_dim=512, num_heads=8, dropout=0.0, hidden_layer_size=2048,
    num_blocks=12, patch_size=8, image_height=64, max_image_width=1024,
)
# (optional knob) you can bump img encoder dropout to ~0.1 for a little extra
# regularization; it has no params so it won't affect load_state_dict.

# --- Build the encoder-decoder (same structure as stage-1) ---
model = EncoderDecoder(
    encoder_config=img_encoder_cfg,
    decoder_config=decoder_config,
    pad_index=tokenizer.pad_token_id,
)

# --- Load the STAGE-1 checkpoint --------------------------------------------
# NOTE: /kaggle/working is wiped between sessions, so upload your stage-1
# final_model.pt as a Kaggle dataset/model and point this at it.
STAGE1_CHECKPOINT = "/kaggle/input/CHANGE-ME/telugu-ocr-stage1/final_model.pt"  # <-- SET THIS
model.load_state_dict(torch.load(STAGE1_CHECKPOINT, map_location="cpu"))

# --- Stage-2 = FULL fine-tuning: everything trainable (no freezing) ---
for _, params in model.named_parameters():
    params.requires_grad = True

# -------------------- sanity: parameter counts --------------------
encoder_params = sum(p.numel() for p in model.encoder_model.parameters())
decoder_params = sum(p.numel() for p in model.decoder_model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"No. of parameters in encoder: {encoder_params}")
print(f"No. of parameters in decoder: {decoder_params}")
print(f"No. of trainable parameters: {trainable_params}")
print(f"Percentage of trainable parameters: {((trainable_params/(encoder_params + decoder_params))*100):.2f}%")  # expect 100.00%


In [ ]:
from datasets import load_dataset

ds = load_dataset("harsha-desaraju/telugu-line-text-image")['train']
ds

In [ ]:
# ds = ds.filter(lambda x: len(x['text'])>3, num_proc=4)
# ds

In [ ]:
split_dataset = ds.train_test_split(test_size=0.01, seed=42)

train_dataset = split_dataset['train']
test_dataset = split_dataset['test']

In [ ]:
PATCH_SIZE = 8
IMAGE_HEIGHT = 64
MAX_IMAGE_WIDTH = 1024

In [ ]:
# ============================================================================
# NEW FOR STAGE-2: target-aware, cost-aware augmentation sampler
# WHERE: new cell; consumed by the TRAIN transform in the next cell.
# WHAT : replaces vit.py's UNIFORM apply_random_augmentation with
#   (a) utility weighting u_i  -> more mass to degradations that look like real
#       photocopies/book scans (roller/drum streaks, bleed-through, ink bleed,
#       low-ink lines, jpeg), less to generic noise (salt-pepper, motion blur);
#   (b) optional cost penalty  -> w_i proportional to u_i / (1 + LAM * t_i);
#       LAM=0 disables it (default);
#   (c) pass-through P_CLEAN    -> a fraction of samples are left CLEAN: trims CPU
#       uniformly AND keeps the model able to read crisp text.
# We deliberately DROPPED the epoch-ramp idea (it fights the cosine LR schedule and
# defers adaptation the model needs early).
#
# RESOLUTION NOTE: augmentation now runs on the 64px-tall crop (ImagePreprocessor
# change), so the full-res times below are pessimistic and the big cost win comes
# from that resolution change -> LAM defaults to 0, utility weighting is the knob.
# PARAM NOTE: scale-dependent augraphy params (roller/drum band widths, bleed
# offsets, ink kernels) were tuned at full res. Eyeball ~a dozen 64px samples and
# retune in vit.py if they look off; the sampler is wrapped so a bad combo at this
# scale just falls back to the clean image (with a printed warning).
# ============================================================================
import random
import numpy as np

# Ensure vit.py is importable (Kaggle: add it as a utility script or place next to
# this notebook) and that augraphy is installed:  pip install augraphy==8.2.6
from vit import COMBOS

# Target-relevance weights in [0, 1]. Higher = looks more like degraded print.
AUG_UTILITY = {
    "DirtyRollers": 1.0, "DirtyDrum": 1.0, "BleedThrough": 0.9, "InkBleed": 0.9,
    "LowInkRandomLines": 0.8, "Jpeg": 0.7, "BrightnessTexturize": 0.6,
    "InkMottling": 0.6, "SubtleNoise": 0.5, "GaussianNoise": 0.5,
    "MotionBlur": 0.3, "SaltPepper": 0.3,
}
# Avg ms/image at FULL res (your measurements). Re-measure at 64px if you set LAM>0.
AUG_TIME_MS = {
    "InkBleed": 1.3, "InkMottling": 0.7, "BleedThrough": 5.4, "LowInkRandomLines": 0.2,
    "BrightnessTexturize": 0.8, "DirtyDrum": 3.9, "DirtyRollers": 16.6, "SubtleNoise": 0.2,
    "Jpeg": 0.3, "MotionBlur": 0.1, "GaussianNoise": 0.6, "SaltPepper": 0.0,
}
assert set(AUG_UTILITY) == set(COMBOS) == set(AUG_TIME_MS), "keys must match COMBOS"

P_CLEAN = 0.25   # fraction left un-augmented (CPU trim + clean-text regularizer)
LAM     = 0.0    # cost-penalty strength per ms; 0 = utility-only. Raise if CPU-bound.

def _build_aug_distribution(utility, times, lam):
    names = list(COMBOS.keys())
    raw = np.array([utility[n] / (1.0 + lam * times[n]) for n in names], dtype=np.float64)
    return names, list(raw / raw.sum())

_AUG_NAMES, _AUG_PROBS = _build_aug_distribution(AUG_UTILITY, AUG_TIME_MS, LAM)

def make_weighted_augmenter(p_clean=P_CLEAN):
    """Return augment(np_uint8 HxWx3) -> np_uint8 HxWx3. Never raises.

    Uses python `random` (torch seeds it per DataLoader worker) so augmentation
    TYPE/severity choices are decorrelated across workers. augraphy's *internal*
    numpy randomness may still correlate across workers (the classic
    numpy-in-DataLoader issue); if you see repeated internal patterns, pass a
    worker_init_fn that reseeds numpy, e.g.:
        def _wi(wid): np.random.seed(torch.initial_seed() % 2**32)
    """
    names, weights = _AUG_NAMES, _AUG_PROBS
    def augment(img):
        if random.random() < p_clean:
            return img
        name = random.choices(names, weights=weights, k=1)[0]
        _label, applier = random.choice(COMBOS[name])   # uniform over the 4 severities
        try:
            out = applier(img)
            if isinstance(out, dict):
                out = out.get("output", img)
            return out
        except Exception as exc:
            print(f"[warn] aug {name} failed at this scale: {exc}; using clean")
            return img
    return augment


In [ ]:
# ============================================================================
# CHANGED FOR STAGE-2
# WHAT : split the single transform into TRAIN (augmented) and EVAL (clean).
# WHERE: was one `sample_transformer` used for BOTH splits.
# WHY  : augmentation must be train-only + on-the-fly; eval must stay fixed/clean
#        so eval_loss / CER are comparable across steps.
# ============================================================================
# Train preprocessor augments the 64px crop; eval preprocessor does not.
train_preprocessor = ImagePreprocessor(IMAGE_HEIGHT, MAX_IMAGE_WIDTH, PATCH_SIZE,
                                       augment_fn=make_weighted_augmenter())
eval_preprocessor  = ImagePreprocessor(IMAGE_HEIGHT, MAX_IMAGE_WIDTH, PATCH_SIZE,
                                       augment_fn=None)


def train_transform(batch):
    images    = [train_preprocessor(img) for img in batch["image"]]
    input_ids = [tokenizer.encode(text)  for text in batch["text"]]
    return {"pixel_values": images, "input_ids": input_ids}


def eval_transform(batch):
    images    = [eval_preprocessor(img) for img in batch["image"]]
    input_ids = [tokenizer.encode(text)  for text in batch["text"]]
    return {"pixel_values": images, "input_ids": input_ids}


In [ ]:
# CHANGED FOR STAGE-2: train gets augmentation, test stays clean.
train_dataset = train_dataset.with_transform(train_transform)
test_dataset = test_dataset.with_transform(eval_transform)


In [ ]:

class OCRCollator:
    """
    Each dataset example is expected to be a dict with:
      - 'pixel_values': float tensor (1, H, W_i)   # H fixed (image_height), W_i variable
      - 'input_ids'   : 1D long tensor / list      # variable length, NO padding yet

    Produces a batch dict whose keys match EncoderDecoder.forward exactly:
      pixel_values      (B, 1, H, W_max)   float
      input_ids         (B, T_max)         long, right-padded with pad_token_id
      img_padding_mask  (B, hp*wp)         float, 1 = PAD patch   (ViT/MHA convention)
      text_padding_mask (B, T_max)         float, 1 = REAL token  (decoder convention)

    NOTE the two opposite conventions are intentional and load-bearing:
      * the image mask feeds nn.MultiheadAttention's key_padding_mask (True = ignore)
      * the text mask feeds the decoder's causal-mask builder (1 = keep)
    Getting either polarity wrong is silent, so they are set explicitly here.

    Assumes every image width W_i is already a multiple of patch_size (your
    ImagePreprocessor pads to that). If not, partial edge patches are counted as real.
    """

    def __init__(self, pad_token_id: int, patch_size: int, image_height: int):
        self.pad_token_id = pad_token_id
        self.patch_size = patch_size
        self.hp = image_height // patch_size  # number of patch ROWS (fixed)

    def __call__(self, batch):
        images = [ex["pixel_values"] for ex in batch]
        # print([len(ex['input_ids']) for ex in batch])
        token_seqs = [torch.as_tensor(ex["input_ids"], dtype=torch.long) for ex in batch]
        B = len(batch)

        # ---- Images ----
        widths = [img.shape[-1] for img in images]
        max_w = max(widths)
        if max_w % self.patch_size != 0:  # safety; should already be a multiple
            max_w += self.patch_size - (max_w % self.patch_size)
        wp = max_w // self.patch_size  # number of patch COLUMNS in the batch

        padded_imgs, img_pad_masks = [], []
        for img, w in zip(images, widths):
            padded_imgs.append(F.pad(img, (0, max_w - w)))  # right-pad width with zeros

            # ceil so a partially-real edge patch is treated as real, not pad
            real_wp = min((w + self.patch_size - 1) // self.patch_size, wp)
            col_pad = torch.zeros(wp, dtype=torch.bool)
            col_pad[real_wp:] = True  # True = padded patch column

            # Conv flattens (hp, wp) row-major -> patch index = h*wp + w.
            # Validity is identical across all hp rows, so tile the column mask.
            img_pad_masks.append(col_pad.unsqueeze(0).expand(self.hp, wp).reshape(-1))

        pixel_values = torch.stack(padded_imgs, dim=0)                  # (B, 1, H, max_w)
        img_padding_mask = torch.stack(img_pad_masks, dim=0).float()    # (B, hp*wp), 1=pad

        # ---- Text (right-pad) ----
        # print([seq.shape for seq in token_seqs])
        max_t = max(seq.shape[0] for seq in token_seqs)
        input_ids = torch.full((B, max_t), self.pad_token_id, dtype=torch.long)
        text_padding_mask = torch.zeros((B, max_t), dtype=torch.float)  # 1 = real
        for i, seq in enumerate(token_seqs):
            n = seq.shape[0]
            input_ids[i, :n] = seq
            text_padding_mask[i, :n] = 1.0

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "img_padding_mask": img_padding_mask,
            "text_padding_mask": text_padding_mask
        }


In [ ]:


# CHANGED FOR STAGE-2: new output dir (don't clobber stage-1), fewer-but-bigger
# effective batches, longer schedule.
OUTPUT_DIR = "/kaggle/working/telugu-ocr-stage2"   # CHANGED (was .../telugu-ocr)
EPOCHS = 4                                          # CHANGED 3 -> 4 (watch CER, extend if improving)
BATCH_SIZE = 16                                     # CHANGED 64 -> 16 (full-FT activation memory on 16GB T4)
GRAD_ACCUM = 4                                      # NEW: effective batch = 16 * 4 * 2 GPUs = 128


data_collator = OCRCollator(pad_token_id=tokenizer.pad_token_id, patch_size=PATCH_SIZE, image_height=IMAGE_HEIGHT)


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # --- Training Duration ---
    num_train_epochs=EPOCHS,

    # --- Batch Size & Accumulation ---
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,          # CHANGED 1 -> 4

    # --- Optimizer & Scheduler ---
    optim="adamw_torch_fused",
    learning_rate=5e-5,                              # CHANGED 3e-4 -> 5e-5 (don't blow up pretrained weights)
    lr_scheduler_type="cosine",
    weight_decay=0.1,
    warmup_ratio=0.1,                                # CHANGED 0.05 -> 0.1 (more warmup for full FT)
    adam_beta2=0.95,
    max_grad_norm=1.0,                               # NEW: explicit clip for full-FT stability

    ddp_find_unused_parameters=False,
    remove_unused_columns=False,

    # --- Precision & Performance ---
    fp16=torch.cuda.is_available(),
    # NOTE: the HF gradient_checkpointing flag is a NO-OP here — EncoderDecoder is a
    # plain nn.Module, not a PreTrainedModel, so Trainer can't call
    # gradient_checkpointing_enable(). If you OOM, drop BATCH_SIZE further, or wrap
    # each transformer block forward with torch.utils.checkpoint yourself.
    gradient_checkpointing=False,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=4,
    dataloader_persistent_workers=True,

    # --- Evaluation & Saving ---
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    # --- Logging ---
    logging_steps=100,
    logging_first_step=True,
    report_to="wandb",
    run_name="stage-2-full-finetuning",              # CHANGED (was stage-1-finetuning)

    # --- Loading best model ---
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",               # NOTE: eval_loss on CLEAN synthetic is
    greater_is_better=False,                         # not the same as CER on degraded books.
    # Once you trust the CER callback below, switch metric_for_best_model="eval_cer".
)


In [ ]:
# ============================================================================
# NEW FOR STAGE-2: lightweight CER eval so you stop judging OCR by eval_loss only.
# WHAT : every CER_EVERY steps, greedily decode CER_SAMPLES *clean* eval images and
#        log character error rate. Defensive: any decode failure warns, never crashes
#        the run. Runs on rank 0 only.
# NOTE : the ideal eval set reflects the TARGET (degraded books). This uses the clean
#        synthetic test split; swap in a fixed degraded/real set when you have one.
# ============================================================================
def _edit_distance(a, b):
    # Levenshtein, dependency-free
    m, n = len(a), len(b)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            cur = dp[j]
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (a[i - 1] != b[j - 1]))
            prev = cur
    return dp[n]


def compute_cer(preds, refs):
    tot_e = tot_c = 0
    for p, r in zip(preds, refs):
        tot_e += _edit_distance(p, r)
        tot_c += max(len(r), 1)
    return tot_e / max(tot_c, 1)


CER_EVERY = 500     # steps between CER evals (independent of eval_steps)
CER_SAMPLES = 64    # eval images to decode per CER eval (B=1 greedy; keep modest)


class CERCallback(TrainerCallback):
    def __init__(self, model, raw_dataset, preprocessor, tokenizer):
        self.model = model
        self.ds = raw_dataset            # UNtransformed split -> has "image" / "text"
        self.prep = preprocessor         # eval_preprocessor (no augmentation)
        self.tok = tokenizer

    def on_step_end(self, args, state, control, **kwargs):
        if not state.is_world_process_zero:
            return
        if state.global_step == 0 or state.global_step % CER_EVERY != 0:
            return
        try:
            was_training = self.model.training
            self.model.eval()
            device = next(self.model.parameters()).device
            n = min(CER_SAMPLES, len(self.ds))
            preds, refs = [], []
            for i in range(n):
                ex = self.ds[i]
                pix = self.prep(ex["image"]).unsqueeze(0).to(device)   # (1, 1, H, W)
                ids = self.model.generate(pix, self.tok.bos_token_id, self.tok.eos_token_id)
                preds.append(self.tok.decode(ids, skip_special_tokens=True))
                refs.append(ex["text"])
            cer = compute_cer(preds, refs)
            print(f"[CER] step {state.global_step}: {cer:.4f}", flush=True)
            try:
                import wandb
                wandb.log({"eval_cer": cer}, step=state.global_step)
            except Exception:
                pass
            if was_training:
                self.model.train()
        except Exception as exc:
            print(f"[CER] skipped at step {state.global_step}: {exc}", flush=True)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    # NEW: raw (untransformed) test split for CER; eval_preprocessor gives clean crops
    callbacks=[CERCallback(model, split_dataset["test"], eval_preprocessor, tokenizer)],
)

trainer.train()

# CHANGED FOR STAGE-2: save under the stage-2 output dir.
torch.save(model.state_dict(), f"{OUTPUT_DIR}/final_model.pt")
print("Finished stage-2 full fine-tuning!")
